# Task 2: Clickbait Spoiler Generation — EDA

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 120

## 1. Load Data

In [ ]:
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

train = load_jsonl('train.jsonl')
val   = load_jsonl('val.jsonl')
test  = load_jsonl('test.jsonl')

print(f'Train: {len(train)} | Val: {len(val)} | Test: {len(test)}')

## 2. Spoiler Text Analysis

In [ ]:
def to_df(records, has_labels=True):
    rows = []
    for r in records:
        row = {
            'postText': r['postText'][0] if r['postText'] else '',
            'targetTitle': r.get('targetTitle', ''),
            'targetDescription': r.get('targetDescription', ''),
            'n_paragraphs': len(r.get('targetParagraphs', [])),
        }
        if has_labels:
            row['spoiler'] = r['spoiler'][0] if r.get('spoiler') else ''
            row['n_spoilers'] = len(r.get('spoiler', []))
            row['spoiler_type'] = r['tags'][0]
        rows.append(row)
    return pd.DataFrame(rows)

train_df = to_df(train)
val_df   = to_df(val)
test_df  = to_df(test, has_labels=False)

print(train_df.head(3))

In [ ]:
# Spoiler length analysis
train_df['spoiler_len'] = train_df['spoiler'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Overall distribution
axes[0].hist(train_df['spoiler_len'].dropna(), bins=50, color='steelblue', edgecolor='black')
axes[0].set_title('Distribution of spoiler length (words)')
axes[0].set_xlabel('Words')
axes[0].set_ylabel('Count')

# By spoiler type
for stype in ['phrase', 'passage', 'multi']:
    subset = train_df[train_df['spoiler_type'] == stype]['spoiler_len'].dropna()
    axes[1].hist(subset, bins=40, alpha=0.6, label=stype)
axes[1].set_title('Spoiler length by type')
axes[1].set_xlabel('Words')
axes[1].legend()

plt.tight_layout()
plt.show()

print('Spoiler length statistics:')
print(train_df.groupby('spoiler_type')['spoiler_len'].describe().round(1))

## 3. Spoiler Uniqueness

In [ ]:
n_unique = train_df['spoiler'].nunique()
n_total = len(train_df)
uniqueness_pct = n_unique / n_total * 100

print(f'Unique spoilers: {n_unique} / {n_total} ({uniqueness_pct:.1f}%)')
print(f'\nMost common spoilers:')
print(train_df['spoiler'].value_counts().head(10))

## 4. Multiple Spoilers per Sample

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Overall
axes[0].hist(train_df['n_spoilers'], bins=range(1, train_df['n_spoilers'].max() + 2), 
             color='coral', edgecolor='black')
axes[0].set_title('Number of spoilers per sample')
axes[0].set_xlabel('Spoiler count')
axes[0].set_ylabel('Samples')

# By type
for stype in ['phrase', 'passage', 'multi']:
    subset = train_df[train_df['spoiler_type'] == stype]['n_spoilers']
    axes[1].hist(subset, bins=range(1, subset.max() + 2), alpha=0.6, label=stype)
axes[1].set_title('Spoiler count by type')
axes[1].set_xlabel('Count')
axes[1].legend()

plt.tight_layout()
plt.show()

print('Spoiler count by type:')
print(train_df.groupby('spoiler_type')['n_spoilers'].value_counts().unstack(fill_value=0))

## 5. Example Spoilers

In [ ]:
for stype in ['phrase', 'passage', 'multi']:
    print(f'\n=== {stype.upper()} ===')
    for _, row in train_df[train_df['spoiler_type'] == stype].head(3).iterrows():
        print(f'  Headline: {row["postText"][:70]}...')
        print(f'  Spoiler : {row["spoiler"]}')
        print()

## 6. Headline vs Spoiler Overlap

In [ ]:
# How much do spoilers repeat words from the headline?
def word_overlap(headline, spoiler):
    if not headline or not spoiler:
        return 0
    h_words = set(headline.lower().split())
    s_words = set(spoiler.lower().split())
    if not s_words:
        return 0
    return len(h_words & s_words) / len(s_words)

train_df['overlap_pct'] = train_df.apply(
    lambda row: word_overlap(row['postText'], row['spoiler']), axis=1
)

fig, ax = plt.subplots(figsize=(9, 4))
for stype in ['phrase', 'passage', 'multi']:
    subset = train_df[train_df['spoiler_type'] == stype]['overlap_pct']
    ax.hist(subset, bins=20, alpha=0.6, label=stype)
ax.set_title('Word overlap: headline → spoiler (%)')
ax.set_xlabel('% of spoiler words in headline')
ax.legend()
plt.tight_layout()
plt.show()

print('Mean overlap by type:')
print(train_df.groupby('spoiler_type')['overlap_pct'].mean().round(3))

## 7. Key Takeaways

- **Spoiler lengths vary widely** — phrase is 1-5 words (median ~3), passage can be 20+ words
- **High uniqueness** — most spoilers appear only once; retrieval baselines will be limited
- **`multi` has multiple spoilers** — multiple target outputs per input
- **Low headline-spoiler overlap** — spoiler is not just copying from headline; requires understanding
- **Baseline strategy**: Retrieval (return most similar training spoiler) will be weak; generation needed